# F-05: Does pruned field-event (management) info help the partial-pool + density + lags model? (RFm)

Field events were tested in F-01 (12-col set → overfit) and F-02 (pruned 2-col set → mild help), but were **dropped from F-03/F-04** (the partial-pooling configs). This re-adds the **pruned tower-specific management** features — `cut` + `manure` recency (exp-decay time-since, D-28) — on top of the F-04 best config and tests whether they add anything.

Hypothesis (cf. F-04 lags): management may be **redundant on the rich base** (gap-filled FCO₂ + density + pooling already encode much ecosystem-state signal).

Two variants, partial pooling (pooled T2+T4+T9 + tower dummies, D-30), density (D-29) + SWC/TS lags (D-31):
- **partial+lags** — F-04 best (reference)
- **partial+lags+mgmt** — + tower-specific cut & manure recency

Per-tower eval (Towers 4/9 std; Tower 2 D-15 split). R-02 gap harness.

In [1]:
from pathlib import Path
import datetime
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
PLAUS_LOW,PLAUS_HIGH=-500,3000
N_REPS,MASK_FRAC=5,0.25
SCENARIOS={"vs":1,"s":4,"m":32,"l":288,"m1":"mixed"}
LSU={"cattle":1.0,"sheep":0.1,"lamb":0.05}
AUX=["_hour_sin","_hour_cos","_doy_sin","_doy_cos"]
LAG_HOURS=[168,336,504,672]
CATCHMENT_AREA_HA={1:4.81,2:6.65,3:6.62,4:7.75,5:6.47,6:3.86,7:2.60,8:7.02,9:7.75,10:1.82,11:1.76,12:1.78,13:1.75,14:1.72,15:1.54}
TOWERS=[2,4,9]

## 1  Configs + load (consolidated + FCO2 + management)

In [2]:
C4="Catchment 4 After  2013/08/13"
def cfg(t,cat,catstr):
    return {"tower":t,"area_ha":CATCHMENT_AREA_HA[cat],
        "target":f"FCH4_1_1_1 [Tower {t}]","ssitc":f"FCH4_SSITC_TEST_1_1_1 [Tower {t}]",
        "sw":f"SWIN_1_1_1 [Tower {t}]","ta":f"TA_0_0_1 [Tower {t}]","vpd":f"VPD_0_0_1 [Tower {t}]",
        "ppfd":f"PPFD_1_1_1 [Tower {t}]","ustar":f"USTAR_0_0_1 [Tower {t}]","ws":f"WS_0_0_1 [Tower {t}]",
        "rn":f"RN_1_1_1 [Tower {t}]","precip":f"Precipitation (mm) [{catstr}]","ts":"TS_1_1_1 [Tower 9]",
        "swc":f"Soil Moisture @ 10cm Depth (%) [{catstr}]","shf":f"SHF_1_1_1 [Tower {t}]","fc":f"FC_1_1_1 [Tower {t}]",
        "livestock":{sp:f"{sp}_{catstr}" for sp in LSU},
        "mgmt_cut":f"mgmt_t{t}_cut_recency","mgmt_manure":f"mgmt_t{t}_manure_recency",
        "train_yrs":[2018] if t==2 else list(range(2018,2022)),
        "test_yrs":[2019] if t==2 else list(range(2022,2024))}
TOWER_CONFIGS={2:cfg(2,2,"Catchment 2"),4:cfg(4,4,C4),9:cfg(9,9,"Catchment 9")}
df=pd.read_csv(HOURLY/"consolidated_hourly.csv",low_memory=False)
df["Datetime"]=pd.to_datetime(df["Datetime"],format="mixed"); df=df.set_index("Datetime")
fco2=pd.read_csv(HOURLY/"fco2_gapfilled.csv",low_memory=False)
fco2["Datetime"]=pd.to_datetime(fco2["Datetime"],format="mixed"); fco2=fco2.set_index("Datetime")
for _t in TOWERS:
    c=f"FC_1_1_1 [Tower {_t}]"
    if c in df.columns: df[c]=fco2[f"FC_gapfilled [Tower {_t}]"]
mg=pd.read_csv(HOURLY/"management_features.csv",low_memory=False)
mg["Datetime"]=pd.to_datetime(mg["Datetime"],format="mixed"); mg=mg.set_index("Datetime")
df=df.join(mg,how="left")
df_raw=df; print("loaded",df_raw.shape)

loaded (70153, 473)


## 2  Functions

In [3]:
def insert_calendar_gaps(d,target,test_yrs,gh,n_reps=N_REPS,seed=0):
    tm=d.index.year.isin(test_yrs); tt=d.index[tm]; va=d.loc[tm,target].notna().values
    n=len(tt); tn=max(1,int(va.sum()*MASK_FRAC)); rb=np.random.default_rng(seed); reps=[]
    for _ in range(n_reps):
        rng=np.random.default_rng(int(rb.integers(0,2**31))); occ=np.zeros(n,bool); m=0
        for sp in rng.permutation(n):
            if m>=tn: break
            g=int(rng.choice([1,4,32,288])) if gh=="mixed" else gh; ep=min(int(sp)+g,n)
            if occ[sp:ep].any(): continue
            occ[sp:ep]=True; m+=int(va[sp:ep].sum())
        reps.append(tt[occ & va])
    return reps

def generic_frame(t):
    c=TOWER_CONFIGS[t]; d=df_raw.copy(); tgt=c["target"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna() & ~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    h,doy=d.index.hour,d.index.dayofyear
    d["_hour_sin"]=np.sin(2*np.pi*h/24); d["_hour_cos"]=np.cos(2*np.pi*h/24)
    d["_doy_sin"]=np.sin(2*np.pi*doy/365); d["_doy_cos"]=np.cos(2*np.pi*doy/365)
    liv=c["livestock"]; area=c["area_ha"]
    lsu=sum(d[liv[s]].fillna(0)*w for s,w in LSU.items())
    g=pd.DataFrame(index=d.index); g["target"]=d[tgt]
    for k in ["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]: g[k]=d[c[k]]
    for a in AUX: g[a]=d[a]
    g["lsu_dens"]=lsu/area
    g["graze"]=(sum(d[liv[s]].fillna(0) for s in LSU)>0).astype(float)
    for lag in LAG_HOURS:
        g[f"swc_lag{lag}"]=g["swc"].shift(lag); g[f"ts_lag{lag}"]=g["ts"].shift(lag)
    g["mgmt_cut"]=d[c["mgmt_cut"]]; g["mgmt_manure"]=d[c["mgmt_manure"]]
    for tt in TOWERS: g[f"is_t{tt}"]=1.0 if tt==t else 0.0
    g["_year"]=d.index.year
    return g

BASE=["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]+AUX+["lsu_dens","graze"]
LAGC=[f"swc_lag{l}" for l in LAG_HOURS]+[f"ts_lag{l}" for l in LAG_HOURS]
MGMT=["mgmt_cut","mgmt_manure"]
DUM=[f"is_t{t}" for t in TOWERS]

def fit(feat,train_df):
    imp=SimpleImputer(strategy="mean")
    rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(train_df[feat].values),train_df["target"].values); return rf,imp
def eval_on(rf,imp,feat,g,c):
    recs=[]; rg={sc:insert_calendar_gaps(g,"target",c["test_yrs"],gh) for sc,gh in SCENARIOS.items()}
    for sc,reps in rg.items():
        for rep,ts in enumerate(reps):
            if len(ts)<5: continue
            yt=g.loc[ts,"target"].values; yp=rf.predict(imp.transform(g.loc[ts,feat].values))
            recs.append({"scenario":sc,"rep":rep,"R2":r2_score(yt,yp)})
    return pd.DataFrame(recs)

## 3  Train (pooled) + evaluate

In [4]:
frames={t:generic_frame(t) for t in TOWERS}
parts=[]
for t in TOWERS:
    g=frames[t]; yrs=TOWER_CONFIGS[t]["train_yrs"]
    sub=g[g["_year"].isin(yrs)]; sub=sub[sub[BASE+["target"]].notna().all(axis=1)]
    parts.append(sub)
pool=pd.concat(parts,ignore_index=True); print("pooled train rows:",len(pool))
rf_lags,imp_lags = fit(BASE+LAGC+DUM, pool)
rf_mg,  imp_mg   = fit(BASE+LAGC+MGMT+DUM, pool)
rows=[]
for t in TOWERS:
    g=frames[t]; c=TOWER_CONFIGS[t]
    for name,(rf,imp,feat) in {
        "partial+lags":(rf_lags,imp_lags,BASE+LAGC+DUM),
        "partial+lags+mgmt":(rf_mg,imp_mg,BASE+LAGC+MGMT+DUM),
    }.items():
        m=eval_on(rf,imp,feat,g,c); m["tower"]=f"Tower {t}"; m["variant"]=name; rows.append(m)
res=pd.concat(rows,ignore_index=True); print("eval rows:",len(res))

pooled train rows: 12558


eval rows: 150


## 4  Results — does pruned management help on the rich base?

In [5]:
VAR=["partial+lags","partial+lags+mgmt"]
for t in TOWERS:
    sub=res[res.tower==f"Tower {t}"]
    tbl=sub.groupby(["variant","scenario"])["R2"].median().unstack("scenario").reindex(index=VAR).reindex(columns=list(SCENARIOS)).round(3)
    print(f"\n=== Tower {t} ==="); print(tbl.to_string())
print("\n--- overall median R2 + management effect ---")
for t in TOWERS:
    s=res[res.tower==f"Tower {t}"].groupby("variant")["R2"].median()
    print(f"Tower {t}: lags {s['partial+lags']:+.3f} -> +mgmt {s['partial+lags+mgmt']:+.3f}  (delta {s['partial+lags+mgmt']-s['partial+lags']:+.3f})")


=== Tower 2 ===
scenario              vs      s      m      l     m1
variant                                             
partial+lags      -0.074 -0.059 -0.102 -0.069 -0.025
partial+lags+mgmt -0.057 -0.049 -0.094 -0.045 -0.018

=== Tower 4 ===
scenario              vs      s      m      l     m1
variant                                             
partial+lags       0.244  0.156  0.244  0.052 -0.098
partial+lags+mgmt  0.242  0.164  0.234  0.050 -0.094

=== Tower 9 ===
scenario              vs      s      m      l     m1
variant                                             
partial+lags       0.291  0.247  0.295  0.200  0.247
partial+lags+mgmt  0.289  0.251  0.295  0.197  0.252

--- overall median R2 + management effect ---
Tower 2: lags -0.062 -> +mgmt -0.049  (delta +0.013)
Tower 4: lags +0.070 -> +mgmt +0.082  (delta +0.012)
Tower 9: lags +0.247 -> +mgmt +0.252  (delta +0.005)


## 5  Append to benchmarks.csv

In [6]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="F-05"]
out=[]
for r in res.to_dict("records"):
    out.append({"replication":"F-05","model":"RFm_"+r["variant"].replace("+","_"),"tower":r["tower"],
        "feature_set":r["variant"],"scenario":r["scenario"],"rep":r["rep"],"split":"pool_T2+4+9",
        "R2":round(r["R2"],4),"date":today,"notes":"F05 add pruned tower mgmt (cut+manure) to partial-pool+density+lags (D-28/D-30/D-31)"})
new=pd.DataFrame(out); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} F-05 rows. Total {len(comb)}.")

Wrote 150 F-05 rows. Total 2515.
